# 🎨 Neural Style Transfer — Dataset Exploration & Starter Guide

Welcome to the **Neural Style Transfer (NST)** dataset inspection and starter notebook!

### 🎯 What is the "Dataset" in Neural Style Transfer?
Unlike standard supervised deep learning tasks (such as image classification or object detection) which require thousands of labeled examples, **Neural Style Transfer (Gatys et al., 2015)** operates primarily on **pairs of single images**:

1. **Content Image ($I_c$)**:
   - Represents the **semantic structure**, shapes, edges, and objects you want to preserve (e.g., a photograph of a building, a bird, or a dancing figure).
2. **Style Image ($I_s$)**:
   - Represents the **artistic texture**, brush strokes, color palette, and geometric patterns (e.g., a painting by Van Gogh or Picasso).

### 🔍 Why Inspect These Files First?
In real-world computer vision projects, image datasets often contain heterogeneous file formats (`.jpg`, `.png`, `.webp`), varying resolutions, and even different color channel modes (`RGB` vs `RGBA` with transparency). 

Feeding unvalidated images into PyTorch or TensorFlow convolutional backbones (like VGG-19, EfficientNet, or ConvNeXt) can lead to:
- **Dimension mismatch errors**: e.g., passing a 4-channel `RGBA` image into a conv layer expecting 3 channels (`RGB`).
- **GPU Out-Of-Memory (OOM)**: e.g., passing multi-megapixel images without dynamic scaling.
- **Aspect ratio distortion**: e.g., naive resizing squashing rectangular compositions.

This notebook performs a complete **pre-flight audit**, verifies compatibility, and provides reusable image loaders ready for any NST model.


## 1. Setup & Environment Verification


In [ ]:
import os
import sys
import glob
from pathlib import Path

# Core Numerical & Visualization
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Check Available Frameworks
torch_available = False
tf_available = False

try:
    import torch
    import torchvision.transforms as transforms
    torch_available = True
    print(f"✅ PyTorch version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
except ImportError:
    print("⚠️ PyTorch not installed in this environment.")

try:
    import tensorflow as tf
    tf_available = True
    print(f"✅ TensorFlow version: {tf.__version__} | GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")
except ImportError:
    print("⚠️ TensorFlow not installed in this environment.")

# Set plot styling
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = False


## 2. Directory Resolution & File Discovery
Let's resolve the repository root and locate the `data/content` and `data/style` folders regardless of where this notebook is executed.


In [ ]:
def resolve_data_paths():
    """Automatically locate content and style directories."""
    possible_roots = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path.cwd() / "data"
    ]
    
    content_dir = None
    style_dir = None
    
    for root in possible_roots:
        cand_content = root / "data" / "content"
        cand_style = root / "data" / "style"
        if cand_content.exists() and cand_style.exists():
            content_dir = cand_content
            style_dir = cand_style
            break
            
    if not content_dir:
        # Fallback check
        if (Path.cwd() / "content").exists():
            content_dir = Path.cwd() / "content"
            style_dir = Path.cwd() / "style"
            
    return content_dir, style_dir

CONTENT_DIR, STYLE_DIR = resolve_data_paths()
print(f"📁 Content Directory : {CONTENT_DIR.resolve()}")
print(f"📁 Style Directory   : {STYLE_DIR.resolve()}")


## 3. Deep-Dive Dataset Audit & Metadata Extraction
Here we scan every single image file and inspect:
- **File Format**: `JPEG`, `PNG`, `WEBP`
- **Color Mode**: `RGB` (3 channels) vs `RGBA` (4 channels with alpha transparency)
- **Dimensions**: Width × Height (pixels)
- **Aspect Ratio**: Width / Height
- **File Size**: In Kilobytes (KB)
- **Uncompressed Memory**: Rough tensor size in RAM


In [ ]:
def inspect_directory(dir_path, category):
    records = []
    supported_exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    
    for file_path in sorted(dir_path.iterdir()):
        if file_path.suffix.lower() in supported_exts and file_path.is_file():
            try:
                with Image.open(file_path) as img:
                    width, height = img.size
                    mode = img.mode
                    channels = len(img.getbands())
                    fmt = img.format
                    file_size_kb = file_path.stat().st_size / 1024
                    aspect_ratio = width / height
                    raw_memory_mb = (width * height * channels) / (1024 * 1024)
                    
                    # Identify warnings
                    flags = []
                    if mode != "RGB":
                        flags.append(f"⚠️ Mode is {mode} (requires RGB conversion)")
                    if width * height > 2_000_000:
                        flags.append("⚠️ High-resolution (>2MP, watch GPU memory)")
                    if aspect_ratio > 1.8 or aspect_ratio < 0.55:
                        flags.append("⚠️ Extreme aspect ratio")
                    
                    status = ", ".join(flags) if flags else "✅ Standard RGB"
                    
                    records.append({
                        "Category": category,
                        "Filename": file_path.name,
                        "Format": fmt,
                        "Mode": mode,
                        "Channels": channels,
                        "Width": width,
                        "Height": height,
                        "Aspect Ratio": round(aspect_ratio, 2),
                        "Size (KB)": round(file_size_kb, 1),
                        "RAM (MB)": round(raw_memory_mb, 2),
                        "Status": status,
                        "Path": str(file_path)
                    })
            except Exception as e:
                records.append({
                    "Category": category,
                    "Filename": file_path.name,
                    "Status": f"❌ Error opening: {e}",
                    "Path": str(file_path)
                })
    return records

content_audit = inspect_directory(CONTENT_DIR, "Content")
style_audit = inspect_directory(STYLE_DIR, "Style")
all_records = content_audit + style_audit

try:
    import pandas as pd
    df = pd.DataFrame(all_records)
    display(df[["Category", "Filename", "Format", "Mode", "Channels", "Width", "Height", "Aspect Ratio", "Size (KB)", "Status"]])
except ImportError:
    for rec in all_records:
        print(f"[{rec['Category']}] {rec['Filename']}: {rec.get('Format')} | {rec.get('Mode')} | {rec.get('Width')}x{rec.get('Height')} | {rec.get('Status')}")


## 4. Key Takeaways from the Dataset Audit

> ### 💡 What You Need to Know About These Files:
> 
> 1. **File Formats**:
>    - **JPEG / PNG**: Standard web image formats.
>    - **WEBP (`first.webp`, `third.webp`)**: Modern, high-efficiency image format developed by Google. PIL (`Pillow`) natively reads WebP, so no special decoder library is needed, but always verify it yields standard RGB arrays.
> 
> 2. **Color Mode Warning (`RGBA` in `Style/ss.png`)**:
>    - `ss.png` has **4 channels** (`RGBA` = Red, Green, Blue, Alpha).
>    - The Alpha channel represents transparency.
>    - ⚠️ **Critical Gotcha**: Neural network backbones (VGG19, EfficientNet, ConvNeXt) expect **exactly 3 color channels**. Passing an RGBA tensor with shape `(1, 4, H, W)` into PyTorch will throw:
>      `RuntimeError: Given groups=1, weight of size [64, 3, 3, 3], expected input[1, 4, H, W] to have 3 channels!`
>    - **Solution**: Always sanitize images with `.convert("RGB")`.
> 
> 3. **High Resolution Warning (`Style/first.webp` = 1800×2352)**:
>    - Gatys' NST optimization requires calculating feature maps and Gram matrices. An 1800×2352 image will require gigabytes of VRAM and easily trigger CUDA Out-Of-Memory.
>    - **Solution**: Scale images so the maximum dimension is between **256 and 512 pixels** during iterative optimization.


## 5. Visual Inspection: Content Images


In [ ]:
def plot_image_grid(records, title, cols=3):
    n = len(records)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    if rows == 1 and cols == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    for idx, rec in enumerate(records):
        ax = axes[idx]
        with Image.open(rec["Path"]) as img:
            ax.imshow(img.convert("RGB"))
            ax.set_title(f"{rec['Filename']}\n{rec['Width']}x{rec['Height']} | {rec['Format']} ({rec['Mode']})", fontsize=10)
            ax.axis("off")
            
    for idx in range(n, len(axes)):
        axes[idx].axis("off")
        
    plt.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

plot_image_grid(content_audit, "📸 Content Images (Structural Target)")


## 6. Visual Inspection: Style Images


In [ ]:
plot_image_grid(style_audit, "🎨 Style Images (Texture & Artistic Target)")


## 7. RGBA vs. RGB Conversion Demonstration
Let's see specifically how `ss.png` (which has 4 channels) is safely converted to 3 channels without losing content.


In [ ]:
rgba_sample = next((r for r in style_audit if r["Mode"] == "RGBA"), None)

if rgba_sample:
    p = rgba_sample["Path"]
    with Image.open(p) as img:
        print(f"Original Mode : {img.mode} | Bands: {img.getbands()}")
        arr_raw = np.array(img)
        print(f"Original Array Shape : {arr_raw.shape}  <-- Note 4 channels!")
        
        # Proper RGB conversion
        img_rgb = img.convert("RGB")
        arr_rgb = np.array(img_rgb)
        print(f"Converted Array Shape: {arr_rgb.shape}  <-- Clean 3 channels!")
        
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))
        ax1.imshow(arr_raw[:, :, :3])
        ax1.set_title("RGB Channels (Color)")
        ax1.axis("off")
        
        ax2.imshow(arr_raw[:, :, 3], cmap="gray")
        ax2.set_title("Alpha Channel (Transparency Mask)")
        ax2.axis("off")
        
        ax3.imshow(arr_rgb)
        ax3.set_title("Sanitized RGB Output")
        ax3.axis("off")
        plt.tight_layout()
        plt.show()


## 8. Robust Preprocessing Function
Here is a complete, production-ready image loader function that handles:
- Any valid file format (`JPEG`, `PNG`, `WEBP`)
- Any color space (auto-converts `RGBA` / Grayscale to `RGB`)
- Aspect-ratio-preserving downscaling (scales largest dimension to `max_dim` e.g., 512px)
- Exports to **PyTorch Tensor**, **TensorFlow Tensor**, or **NumPy Array**


In [ ]:
def load_and_preprocess_image(image_path, max_dim=512, framework="numpy"):
    """
    Robust image loader for Neural Style Transfer pipelines.
    
    Args:
        image_path: Path to the image file (JPG, PNG, WEBP, etc.)
        max_dim: Maximum width or height; scales down while preserving aspect ratio.
        framework: 'numpy', 'pytorch', or 'tensorflow'
        
    Returns:
        Preprocessed image tensor/array ready for model forward pass.
    """
    # 1. Open with PIL and enforce 3-channel RGB
    img = Image.open(image_path).convert("RGB")
    
    # 2. Rescale while preserving aspect ratio
    orig_w, orig_h = img.size
    scale = max_dim / max(orig_w, orig_h)
    if scale < 1.0:  # Only downscale, do not artificially upscale tiny images
        new_w = int(round(orig_w * scale))
        new_h = int(round(orig_h * scale))
        img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
        
    arr = np.array(img, dtype=np.float32) / 255.0  # Normalize to [0, 1]
    
    if framework.lower() == "numpy":
        return arr  # Shape: (H, W, 3)
        
    elif framework.lower() in ("pytorch", "torch"):
        if not torch_available:
            raise RuntimeError("PyTorch is not installed.")
        # PyTorch format: (Batch, Channel, Height, Width) -> (1, 3, H, W)
        tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).float()
        return tensor
        
    elif framework.lower() in ("tensorflow", "tf"):
        if not tf_available:
            raise RuntimeError("TensorFlow is not installed.")
        # TensorFlow format: (Batch, Height, Width, Channel) -> (1, H, W, 3)
        tensor = tf.convert_to_tensor(arr, dtype=tf.float32)
        tensor = tf.expand_dims(tensor, axis=0)
        return tensor
        
    else:
        raise ValueError(f"Unknown framework '{framework}'. Choose 'numpy', 'pytorch', or 'tensorflow'.")

# Test on a WEBP style image and a standard JPG content image
test_content_path = content_audit[0]["Path"]
test_style_path = next(r["Path"] for r in style_audit if r["Format"] == "WEBP")

print(f"Testing Content : {Path(test_content_path).name}")
print(f"Testing Style   : {Path(test_style_path).name}")

if torch_available:
    t_content = load_and_preprocess_image(test_content_path, max_dim=512, framework="pytorch")
    t_style = load_and_preprocess_image(test_style_path, max_dim=512, framework="pytorch")
    print(f"✅ PyTorch Content Tensor Shape : {t_content.shape} | Range: [{t_content.min():.2f}, {t_content.max():.2f}]")
    print(f"✅ PyTorch Style Tensor Shape   : {t_style.shape} | Range: [{t_style.min():.2f}, {t_style.max():.2f}]")

if tf_available:
    tf_content = load_and_preprocess_image(test_content_path, max_dim=512, framework="tensorflow")
    tf_style = load_and_preprocess_image(test_style_path, max_dim=512, framework="tensorflow")
    print(f"✅ TensorFlow Content Tensor Shape: {tf_content.shape} | Range: [{tf.reduce_min(tf_content):.2f}, {tf.reduce_max(tf_content):.2f}]")
    print(f"✅ TensorFlow Style Tensor Shape  : {tf_style.shape} | Range: [{tf.reduce_min(tf_style):.2f}, {tf.reduce_max(tf_style):.2f}]")


## 9. Interactive Pair Preview
Pick any content and style image to preview how they pair up before passing them into the neural transfer models!


In [ ]:
# Choose your test pair
SELECTED_CONTENT = CONTENT_DIR / "bird.jpg"
SELECTED_STYLE = STYLE_DIR / "second.jpg"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

with Image.open(SELECTED_CONTENT) as c_img:
    ax1.imshow(c_img.convert("RGB"))
    ax1.set_title(f"Target Content: {SELECTED_CONTENT.name}\nSize: {c_img.size}", fontsize=11)
    ax1.axis("off")

with Image.open(SELECTED_STYLE) as s_img:
    ax2.imshow(s_img.convert("RGB"))
    ax2.set_title(f"Target Style: {SELECTED_STYLE.name}\nSize: {s_img.size}", fontsize=11)
    ax2.axis("off")

plt.suptitle("✨ Input Pair for Neural Style Transfer", fontsize=13)
plt.tight_layout()
plt.show()


## 🚀 Next Steps: Running Style Transfer Models

Now that the dataset is verified and sanitized, you can explore the models available in this repository:

### 🐍 PyTorch Notebooks (`notebooks/pytorch/`):
- **`01_vgg19_nst.ipynb`**: Gatys et al. classical optimization using VGG-19 features.
- **`02_efficientnet_b0_nst.ipynb`**: Experimental NST using lightweight EfficientNet-B0 backbone.
- **`03_levit_nst.ipynb`**: Vision Transformer hybrid (LeViT) for attention-guided style transfer.

### 🔶 TensorFlow Notebooks (`notebooks/tensorflow/`):
- **`01_vgg19_nst.ipynb`**: Standard Keras VGG-19 implementation.
- **`02_gatys_baseline_nst.ipynb`**: Official Gatys baseline implementation.
- **`03_gatys_custom_scratch_nst.ipynb`**: Custom loss formulation and Gram matrix from scratch.
- **`06_convnext_nst.ipynb`**: Modern ConvNeXt architecture for style transfer.
- **`08_paper_based_segmentation_nst.ipynb`**: Segmentation-guided NST using foreground mask separation.

### 🔀 Hybrid Fusion (`notebooks/fusion/`):
- **`01_fusion_model_nst.ipynb`**: Multi-backbone ensemble combining content and style losses from multiple networks.

Happy Stylizing! 🎨
